In [ ]:
# 📦 Install packages
!pip install rdflib networkx matplotlib gradio

In [ ]:
# 📚 Imports
import rdflib
from rdflib import Graph, RDF, Namespace, Literal
import networkx as nx
import matplotlib.pyplot as plt
import gradio as gr
from io import BytesIO


In [ ]:
# 🌐 Build RDF graph
EX = Namespace("http://example.org/")
g = Graph()
g.bind("ex", EX)

g.add((EX.Cancer, RDF.type, EX.Disease))
g.add((EX.Cancer, EX.hasTreatment, EX.Chemotherapy))
g.add((EX.Chemotherapy, EX.sideEffect, Literal("Hair Loss")))
g.add((EX.Chemotherapy, EX.sideEffect, Literal("Fatigue")))


In [ ]:
# 🧠 Convert RDFLib to NetworkX
def rdf_to_networkx(rdf_graph):
    nxg = nx.DiGraph()
    for subj, pred, obj in rdf_graph:
        s = str(subj).split("/")[-1]
        p = str(pred).split("/")[-1]
        o = str(obj).split("/")[-1] if isinstance(obj, rdflib.URIRef) else str(obj)
        nxg.add_edge(s, o, label=p)
    return nxg


In [ ]:
# 📊 Draw graph
def draw_graph():
    G = rdf_to_networkx(g)
    pos = nx.spring_layout(G, seed=42)
    edge_labels = nx.get_edge_attributes(G, 'label')

    fig, ax = plt.subplots(figsize=(8, 6))
    nx.draw(G, pos, with_labels=True, node_color='skyblue', edge_color='gray', node_size=2500, font_size=10, ax=ax)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', ax=ax)

    buf = BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    return buf


In [ ]:
# 🎨 Gradio App
with gr.Blocks() as demo:
    gr.Markdown("## 🧬 RDF Knowledge Graph Visualizer")
    img_output = gr.Image(type="filepath")
    btn = gr.Button("Visualize KG")
    btn.click(fn=draw_graph, inputs=[], outputs=img_output)

demo.launch()
